# SQL Worksheet — Week4

Use the following tables from the Riva Data Platform:

- `rivadataplatform.dataproduct.dim_batch`
- `rivadataplatform.dataproduct.dim_class`
- `rivadataplatform.dataproduct.fact_attendance`
- `rivadataplatform.dataproduct.dim_student`
- `rivadataplatform.dataproduct.dim_date`

**Instructions**
- Write SQL for each question.
- Do not modify the source data.
- Use clear aliases where JOINs are involved.
- Unless a question specifically asks for a particular column, select only the columns needed to answer it.


## Tables / Relationships

Useful keys:
- `dim_student.student_key` ↔ `fact_attendance.student_key`
- `dim_class.class_key` ↔ `fact_attendance.class_key`
- `dim_batch.batch_key` ↔ `fact_attendance.batch_key`
- `dim_class.batch_id` ↔ `dim_batch.batch_id`

## Question 1 — Complete Attendance Detail
Join all five tables and return one row per attendance record with student identity, null-safe city, batch, class date, calendar day, topic, class status, attendance status, and remarks. Replace null remarks and topics with labels and sort by date and student.

In [ ]:
SELECT
    f.attendance_id,
    s.student_id,
    s.student_name,
    COALESCE(s.city, 'Unknown city') AS city,
    b.batch_name,
    c.class_date,
    COALESCE(d.day_name, c.class_day) AS day_name,
    COALESCE(c.topic, 'Topic not assigned') AS topic,
    c.status AS class_status,
    f.attendance_status,
    COALESCE(f.remarks, 'No remarks') AS remarks
FROM rivadataplatform.dataproduct.fact_attendance AS f
JOIN rivadataplatform.dataproduct.dim_student AS s
    ON s.student_key = f.student_key
JOIN rivadataplatform.dataproduct.dim_class AS c
    ON c.class_key = f.class_key
JOIN rivadataplatform.dataproduct.dim_batch AS b
    ON b.batch_key = f.batch_key
LEFT JOIN rivadataplatform.dataproduct.dim_date AS d
    ON d.date_key = f.date_key
ORDER BY c.class_date, s.student_name;

## Question 2 — Class Attendance Scorecard
For every class, including classes with no attendance, join class, batch, attendance, and student dimensions. Group by class and return distinct students, Present/Late/Absent counts, and total records. Label null topics and preserve zero counts.

In [ ]:
SELECT
    c.class_id,
    COALESCE(c.topic, 'Topic not assigned') AS topic,
    b.batch_name,
    COUNT(DISTINCT f.student_key) AS distinct_students,
    COUNT(f.attendance_id) AS total_records,
    SUM(CASE WHEN f.attendance_status = 'Present' THEN 1 ELSE 0 END) AS present_count,
    SUM(CASE WHEN f.attendance_status = 'Late' THEN 1 ELSE 0 END) AS late_count,
    SUM(CASE WHEN f.attendance_status = 'Absent' THEN 1 ELSE 0 END) AS absent_count
FROM rivadataplatform.dataproduct.dim_class AS c
LEFT JOIN rivadataplatform.dataproduct.fact_attendance AS f
    ON f.class_key = c.class_key
LEFT JOIN rivadataplatform.dataproduct.dim_batch AS b
    ON b.batch_id = c.batch_id
GROUP BY c.class_id, COALESCE(c.topic, 'Topic not assigned'), b.batch_name
ORDER BY c.class_id;

## Question 3 — Attendance Rate by Class Date
Join class, batch, attendance, and date dimensions. For each class date and topic, calculate total records, attended records (`Present` + `Late`), absent records, and attendance rate. Use `NULLIF`, label null topics, and return only dates with attendance.

In [ ]:
SELECT
    c.class_date,
    COALESCE(d.day_name, c.class_day) AS day_name,
    b.batch_name,
    COALESCE(c.topic, 'Topic not assigned') AS topic,
    COUNT(f.attendance_id) AS total_records,
    SUM(CASE WHEN f.attendance_status IN ('Present', 'Late') THEN 1 ELSE 0 END) AS attended_records,
    SUM(CASE WHEN f.attendance_status = 'Absent' THEN 1 ELSE 0 END) AS absent_records,
    100.0 * SUM(CASE WHEN f.attendance_status IN ('Present', 'Late') THEN 1 ELSE 0 END)
        / NULLIF(COUNT(f.attendance_id), 0) AS attendance_rate
FROM rivadataplatform.dataproduct.dim_class AS c
JOIN rivadataplatform.dataproduct.fact_attendance AS f
    ON f.class_key = c.class_key
JOIN rivadataplatform.dataproduct.dim_batch AS b
    ON b.batch_key = f.batch_key
LEFT JOIN rivadataplatform.dataproduct.dim_date AS d
    ON d.date_key = f.date_key
GROUP BY c.class_date, COALESCE(d.day_name, c.class_day), b.batch_name, COALESCE(c.topic, 'Topic not assigned')
HAVING COUNT(f.attendance_id) > 0
ORDER BY c.class_date;

## Question 4 — Classes With Absence Risk
Join classes, batches, attendance, and students. Group by class, return absent count, distinct affected students, and absent percentage, and keep only classes where the absent count is greater than zero. Include null-safe topic text.

In [ ]:
SELECT
    c.class_id,
    COALESCE(c.topic, 'Topic not assigned') AS topic,
    b.batch_name,
    COUNT(DISTINCT CASE WHEN f.attendance_status = 'Absent' THEN f.student_key END) AS affected_students,
    SUM(CASE WHEN f.attendance_status = 'Absent' THEN 1 ELSE 0 END) AS absent_count,
    100.0 * SUM(CASE WHEN f.attendance_status = 'Absent' THEN 1 ELSE 0 END)
        / NULLIF(COUNT(f.attendance_id), 0) AS absent_percentage
FROM rivadataplatform.dataproduct.dim_class AS c
JOIN rivadataplatform.dataproduct.fact_attendance AS f
    ON f.class_key = c.class_key
JOIN rivadataplatform.dataproduct.dim_student AS s
    ON s.student_key = f.student_key
JOIN rivadataplatform.dataproduct.dim_batch AS b
    ON b.batch_key = f.batch_key
GROUP BY c.class_id, COALESCE(c.topic, 'Topic not assigned'), b.batch_name
HAVING SUM(CASE WHEN f.attendance_status = 'Absent' THEN 1 ELSE 0 END) > 0
ORDER BY absent_count DESC;

## Question 5 — Student Issues by Location
Join students, attendance, classes, batches, and dates. Group by student and null-safe city, count Late and Absent records, calculate the issue rate, and return only students with at least one issue. Order by issue rate descending.

In [ ]:
SELECT
    s.student_id,
    s.student_name,
    COALESCE(s.city, 'Unknown city') AS city,
    COALESCE(MAX(b.batch_name), 'No batch') AS batch_name,
    COUNT(f.attendance_id) AS total_records,
    SUM(CASE WHEN f.attendance_status IN ('Late', 'Absent') THEN 1 ELSE 0 END) AS problem_attendance_count,
    100.0 * SUM(CASE WHEN f.attendance_status IN ('Late', 'Absent') THEN 1 ELSE 0 END)
        / NULLIF(COUNT(f.attendance_id), 0) AS issue_rate
FROM rivadataplatform.dataproduct.dim_student AS s
JOIN rivadataplatform.dataproduct.fact_attendance AS f
    ON f.student_key = s.student_key
JOIN rivadataplatform.dataproduct.dim_class AS c
    ON c.class_key = f.class_key
LEFT JOIN rivadataplatform.dataproduct.dim_batch AS b
    ON b.batch_key = f.batch_key
LEFT JOIN rivadataplatform.dataproduct.dim_date AS d
    ON d.date_key = f.date_key
GROUP BY s.student_id, s.student_name, COALESCE(s.city, 'Unknown city')
HAVING SUM(CASE WHEN f.attendance_status IN ('Late', 'Absent') THEN 1 ELSE 0 END) > 0
ORDER BY issue_rate DESC, s.student_name;

## Question 6 — Batch Relationship Data Quality
Join attendance to class and both batch representations. Group by class and recorded/expected batch names, then return only mismatches between the attendance `batch_key` and the batch identified by `dim_class.batch_id`. Include mismatch record counts and null-safe labels.

In [ ]:
SELECT
    c.class_id,
    COALESCE(c.topic, 'Topic not assigned') AS topic,
    f.batch_key AS recorded_batch_key,
    expected_batch.batch_key AS expected_batch_key,
    COALESCE(recorded_batch.batch_name, 'Recorded batch not found') AS recorded_batch_name,
    COALESCE(expected_batch.batch_name, 'Expected batch not found') AS expected_batch_name,
    COUNT(f.attendance_id) AS mismatch_records
FROM rivadataplatform.dataproduct.fact_attendance AS f
JOIN rivadataplatform.dataproduct.dim_class AS c
    ON c.class_key = f.class_key
LEFT JOIN rivadataplatform.dataproduct.dim_batch AS recorded_batch
    ON recorded_batch.batch_key = f.batch_key
LEFT JOIN rivadataplatform.dataproduct.dim_batch AS expected_batch
    ON expected_batch.batch_id = c.batch_id
GROUP BY c.class_id, COALESCE(c.topic, 'Topic not assigned'), f.batch_key,
    expected_batch.batch_key, COALESCE(recorded_batch.batch_name, 'Recorded batch not found'),
    COALESCE(expected_batch.batch_name, 'Expected batch not found')
HAVING f.batch_key IS DISTINCT FROM expected_batch.batch_key
ORDER BY c.class_id;